In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("hf://datasets/daniilak/Russia_Real_Estate_2018_2021/data.csv")


print("First 5 records:", df.head())


print("То, что выдает команда df.info: ",df.info())
print("То, что выдает команда df.describe: ",df.describe())
print("То, что выдает команда df.shape: ", df.shape)

First 5 records:      price        date      time    geo_lat    geo_lon  region  building_type  \
0  6050000  2018-02-19  20:00:21  59.805808  30.376141    2661              1   
1  8650000  2018-02-27  12:04:54  55.683807  37.297405      81              3   
2  4000000  2018-02-28  15:44:00  56.295250  44.061637    2871              1   
3  1850000  2018-03-01  11:24:52  44.996132  39.074783    2843              4   
4  5450000  2018-03-01  17:42:43  55.918767  37.984642      81              3   

   level  levels  rooms  area  kitchen_area  object_type  
0      8      10      3  82.6          10.8            1  
1      5      24      2  69.1          12.0            1  
2      5       9      3  66.0          10.0            1  
3     12      16      2  38.0           5.0           11  
4     13      14      2  60.0          10.0            1  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5477006 entries, 0 to 5477005
Data columns (total 13 columns):
 #   Column         Dtype  
-

In [3]:
print(df.isnull().sum())

print("Duplicates: ", df.duplicated().sum())
df = df.drop_duplicates()
print("Reamining duplicates: ", df.duplicated().sum())

price            0
date             0
time             0
geo_lat          0
geo_lon          0
region           0
building_type    0
level            0
levels           0
rooms            0
area             0
kitchen_area     0
object_type      0
dtype: int64
Duplicates:  1523
Reamining duplicates:  0


In [4]:
print("Минимумы и максимумы: ", df.describe())

Минимумы и максимумы:                price       geo_lat       geo_lon        region  building_type  \
count  5.475483e+06  5.475483e+06  5.475483e+06  5.475483e+06   5.475483e+06   
mean   4.421478e+06  5.403784e+01  5.324654e+01  4.307440e+03   1.949033e+00   
std    2.151016e+07  4.622977e+00  2.074788e+01  3.307991e+03   1.038568e+00   
min   -2.144967e+09  4.145906e+01  1.989020e+01  3.000000e+00   0.000000e+00   
25%    1.950000e+06  5.337726e+01  3.777797e+01  2.661000e+03   1.000000e+00   
50%    2.990000e+06  5.517128e+01  4.307021e+01  2.922000e+03   2.000000e+00   
75%    4.800020e+06  5.622613e+01  6.564969e+01  6.171000e+03   3.000000e+00   
max    2.147484e+09  7.198040e+01  1.625361e+02  6.188800e+04   5.000000e+00   

              level        levels         rooms          area  kitchen_area  \
count  5.475483e+06  5.475483e+06  5.475483e+06  5.475483e+06  5.475483e+06   
mean   6.213816e+00  1.139749e+01  1.726213e+00  5.391823e+01  1.062817e+01   
std    4.956792e+00

In [5]:
#Начинаем чистить
df = df[df['price'] > 0]
df = df[df['rooms'] > 0]
df = df[df['area'] > 10]
df = df[df['area'] < 600]
df = df[df['kitchen_area'] <= df['area']]
df = df[df['kitchen_area'] < 150]
df = df[df['level'] <= df['levels']]

In [6]:
df = df.sample(500000, random_state=42)


In [7]:
print(df.columns)

Index(['price', 'date', 'time', 'geo_lat', 'geo_lon', 'region',
       'building_type', 'level', 'levels', 'rooms', 'area', 'kitchen_area',
       'object_type'],
      dtype='object')


In [8]:
y = df['price']

In [9]:
df['date'] = pd.to_datetime(df['date'])

df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

df = df.drop(['date', 'time'], axis=1)


In [10]:
df['floor_ratio'] = df['level']/df['levels']
df['kitchen_ratio'] = df['kitchen_area']/df['area']


In [11]:
df = pd.get_dummies(df, columns=['region', 'building_type', 'object_type'], drop_first=True)

In [12]:
y = np.log1p(df['price'])
x = df.drop('price', axis=1)
print(x.isnull().sum().sum())

0


In [13]:
print(x.shape)

(500000, 100)


In [14]:
#y = np.log1p(df['price'])

In [15]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [16]:
model = LinearRegression()
model.fit(x_train, y_train)

y_pred_log = model.predict(x_test)

In [17]:
y_pred = np.expm1(y_pred_log)
y_test_real = np.expm1(y_test)

In [18]:
from sklearn.metrics import mean_squared_error, r2_score

mse = mean_squared_error(y_test_real, y_pred)
r2 = r2_score(y_test_real, y_pred)

rmse = np.sqrt(mse)

print("MSE: ", mse)
print("R2: ", r2)
print("RMSE: ", rmse)

MSE:  128174444311023.92
R2:  0.012669411710264367
RMSE:  11321415.296288


In [22]:
from sklearn.metrics import median_absolute_error
print(median_absolute_error(y_test_real, y_pred))

from sklearn.metrics import mean_squared_log_error

rmsle = np.sqrt(mean_squared_log_error(y_test_real, y_pred))
print("RMSLE:", rmsle)

print("R2 (log space):", r2_score(y_test, y_pred_log))
print("Median AE:", median_absolute_error(y_test_real, y_pred))


595252.7246851176
RMSLE: 0.3909900359013295
R2 (log space): 0.7178041545118226
Median AE: 595252.7246851176


In [19]:
coef_df = pd.DataFrame({'feature': x.columns, 'coef': model.coef_})

coef_df = coef_df.sort_values(by='coef', ascending=False)

print(coef_df.head(10))
print(coef_df.tail(10))


         feature      coef
87  region_13098  5.326897
92  region_16705  5.323510
84  region_11171  3.958558
43   region_4086  3.640318
16   region_1901  3.609590
73   region_7929  3.593667
11     region_69  2.943923
50   region_4963  2.863178
81  region_10160  2.532963
78   region_9579  2.183345
         feature      coef
28   region_2843 -1.992582
42   region_4007 -1.995484
75   region_8509 -2.039735
33   region_2900 -2.049939
89  region_13919 -2.122832
86  region_11991 -2.160296
79   region_9648 -2.184057
32   region_2885 -2.188942
85  region_11416 -2.226096
88  region_13913 -2.524218


In [ ]:


#print(coef_df.head(10))
#print(coef_df.tail(10))